In [1]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import List

In [2]:
import os
os.getcwd()


'/home/suyash/codes/PINN/research'

In [3]:
os.chdir("../")

In [4]:
@dataclass
class GeotechnicalParamsConfig:
    Ks: float 
    theta_s: float 
    theta_r: float 
    alpha: float 
    n: float 
    l: float 
    c_prime: float
    phi_prime: float 
    gamma: float 
    beta: float 
    depth: float 




In [5]:
import yaml
def _load_yaml(path: Path) -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)
params_path = Path("params.yaml")
print(f"Loading parameters from {params_path}")
params = _load_yaml(params_path)

Loading parameters from params.yaml


In [6]:

def get_geotechnical_params() -> GeotechnicalParamsConfig:
        
        g = params["geotechnical"]
        return GeotechnicalParamsConfig(**g)
get_geotechnical_params()


GeotechnicalParamsConfig(Ks=1.5e-05, theta_s=0.45, theta_r=0.05, alpha=0.8, n=1.4, l=0.5, c_prime=5.0, phi_prime=28.0, gamma=20.0, beta=42.0, depth=25.0)

In [7]:
import torch
import torch.nn as nn
import math
from typing import Dict, Optional

In [12]:
#math functions 
# ── Van Genuchten helpers (PyTorch — differentiable) ─────────────────────────

def _vg_Se(psi: torch.Tensor, alpha: float, n: float) -> torch.Tensor:
    """Effective saturation Se(psi). psi in metres."""
    m = 1.0 - 1.0 / n
    return torch.where(
        psi < 0,
        (1.0 + (alpha * torch.abs(psi)) ** n) ** (-m),
        torch.ones_like(psi),
    )


def _vg_K(psi: torch.Tensor, Ks: float, alpha: float, n: float, l: float = 0.5) -> torch.Tensor:
    m = 1.0 - 1.0 / n
    Se = _vg_Se(psi, alpha, n).clamp(1e-8, 1.0)
    inner = (1.0 - (1.0 - Se ** (1.0 / m)) ** m) ** 2
    return Ks * Se ** l * inner


def _vg_C(psi: torch.Tensor, theta_r: float, theta_s: float, alpha: float, n: float) -> torch.Tensor:
    """Specific moisture capacity C = dtheta/dpsi."""
    m = 1.0 - 1.0 / n
    C = torch.where(
        psi < 0,
        (theta_s - theta_r) * m * n * alpha
        * (alpha * torch.abs(psi)) ** (n - 1)
        / (1.0 + (alpha * torch.abs(psi)) ** n) ** (m + 1),
        torch.zeros_like(psi),
    )
    return C.clamp(min=1e-8)


In [8]:
@dataclass
class LossBreakdown:
    """Data class to track individual loss components during training."""
    physics: torch.Tensor
    anchor: torch.Tensor
    initial: torch.Tensor
    boundary: torch.Tensor
    failure: torch.Tensor
    total: torch.Tensor

    def as_dict(self) -> Dict[str, float]:
        return {k: v.item() for k, v in self.__dict__.items()}